# 04 - Model Evaluation
Evaluates the trained models on the held-out chronological test split using imbalance-aware metrics (precision, recall, F1, ROC-AUC, PR-AUC, confusion matrix).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "04_Source_Code"))
import config
from utils import get_logger
logger = get_logger("notebook")
print("Project root:", config.PROJECT_ROOT)
print("Study region:", config.STUDY_REGION["name"])


In [ ]:
from data_loader import load_dataset
from train_models import train_all_models
from evaluate_models import evaluate_all_models, best_model_name

df = load_dataset(source="demo")
result = train_all_models(df)
comparison = evaluate_all_models(result)
comparison


## Why not judge by accuracy alone?
Wildfire occurrence is rare (a few percent of grid-weeks). A trivial model that always predicts 'no fire' would score >95% accuracy while catching zero real fires. **Precision, recall, F1, ROC-AUC, and especially PR-AUC** are reported because they are far more informative under class imbalance.

In [ ]:
from visualization import plot_model_comparison, plot_confusion_matrix
plot_model_comparison(comparison)

from sklearn.metrics import confusion_matrix
best = best_model_name(comparison)
best_bundle = result["models"][best]
X_test, y_test = result["splits"]["test"]
X_in = best_bundle["scaler"].transform(X_test) if best_bundle.get("uses_scaler") else X_test
y_pred = (best_bundle["model"].predict_proba(X_in)[:,1] >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)
plot_confusion_matrix(cm, best)
print("Best model:", best)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
fig, ax = plt.subplots(figsize=(9,5))
ax.imshow(mpimg.imread(config.PLOTS_DIR / "model_comparison.png"))
ax.axis("off")
plt.show()


## Illustrative results only
These metrics describe performance on the **synthetic demo dataset** and are not evidence of real-world predictive skill. Real-world validation requires NASA FIRMS / weather / satellite data (future work).